# Setup: environment check

Run the cells below to check Python, PhysicsNeMo 2.2.2, and symbolic derivatives in the selected kernel.

The default device setting uses CUDA when available. `AI4SCI_DEVICE=cpu` selects the CPU; `AI4SCI_DEVICE=cuda` requires a working GPU.

[Start Here](Start_Here.ipynb) · Next: [Introduction](01_Introduction.ipynb)


In [ ]:
from pathlib import Path
import importlib.metadata as metadata
import json
import platform

import os
REQUESTED_DEVICE = os.environ.get("AI4SCI_DEVICE", "auto").lower()
if REQUESTED_DEVICE not in {"auto", "cpu", "cuda"}:
    raise ValueError("AI4SCI_DEVICE must be auto, cpu, or cuda.")
REQUIRE_CUDA = REQUESTED_DEVICE == "cuda"
EXPECTED_PHYSICSNEMO = "2.2.2"
root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "Start_Here.ipynb").is_file()), None)
if root is None:
    raise RuntimeError("Open this notebook inside the repository; Start_Here.ipynb was not found.")
print("Python:", platform.python_version())
version = metadata.version("nvidia-physicsnemo")
print("PhysicsNeMo:", version)
if version != EXPECTED_PHYSICSNEMO:
    raise RuntimeError(f"This course requires version {EXPECTED_PHYSICSNEMO}. Select the designated environment/kernel.")


In [ ]:
import torch
from physicsnemo.models.mlp import FullyConnected
from physicsnemo.models.fno import FNO
from physicsnemo.models.afno import AFNO
from physicsnemo.sym.eq.pde import PDE
from physicsnemo.sym.eq.phy_informer import PhysicsInformer

cuda_available = torch.cuda.is_available()
if REQUIRE_CUDA and not cuda_available:
    raise RuntimeError("GPU check requested, but no CUDA GPU is available.")
device = torch.device("cuda" if REQUIRE_CUDA or (REQUESTED_DEVICE == "auto" and cuda_available) else "cpu")
print("PyTorch:", torch.__version__)
print("Selected device:", device)
if cuda_available:
    print("GPU:", torch.cuda.get_device_name())
torch.manual_seed(42)
network = FullyConnected(in_features=1, out_features=1, num_layers=2, layer_size=16).to(device)
x = torch.linspace(0, 1, 8, device=device).reshape(-1, 1)
y = network(x)
loss = y.square().mean()
loss.backward()
assert torch.isfinite(y).all()
assert all(p.grad is None or torch.isfinite(p.grad).all() for p in network.parameters())
print("Neural-network forward/backward check passed:", tuple(y.shape))


In [ ]:
from sympy import Function, Symbol

class PoissonCheck(PDE):
    def __init__(self):
        self.dim = 1
        x = Symbol("x")
        u = Function("u")(x)
        self.equations = {"check": u.diff(x, 2) - 2}

coordinates = torch.linspace(0, 1, 8, device=device).reshape(-1, 1).requires_grad_(True)
u = coordinates.square()
informer = PhysicsInformer(["check"], PoissonCheck(), grad_method="autodiff", device=device)
residual = informer.forward({"coordinates": coordinates, "u": u})["check"]
max_residual = residual.detach().abs().max().item()
assert max_residual < 1e-5
print("Symbolic residual check passed: u=x² gives u_xx-2=0")
print("Inspect each lesson run separately for training quality and convergence.")


In [ ]:
manifest = json.loads((root / "ETC/course_materials/course_manifest.json").read_text())
required = [item["notebook"] for item in manifest["course"]]
required += [item["script"] for item in manifest["runs"]]
missing_files = sorted({path for path in required if not (root / path).is_file()})
if missing_files:
    raise FileNotFoundError("Required course files are missing: " + ", ".join(missing_files))
print(f"Verified {len(manifest['course'])} required notebooks and all program paths.")


## Next lesson

After the checks pass, read the [Introduction](01_Introduction.ipynb), then open [Lab 1: first problem](01_labs/01_pinn/Lab_1_PINN_Fundamentals.ipynb#first-problem).

Lab 1 starts with $u_{xx}=1$ and $u(0)=u(1)=0$. Run its [setup, training, and plotting cells](01_labs/01_pinn/Lab_1_PINN_Fundamentals.ipynb#forward-pinn-execution) in order.

[Back to Start Here](Start_Here.ipynb)

If a check fails, confirm the selected kernel before installing packages. Saved outputs may come from an earlier session; rerun the cells to check this machine.